In [0]:
%run  ../00-common/01_Environmnet_config

In [0]:
silver_results_table = f"{catalog_name}.{silver_schema}.results"
silver_sprints_table = f"{catalog_name}.{silver_schema}.sprints"
target_table = f"{catalog_name}.{gold_schema}.fact_results"

In [0]:
results_df = (
    spark.read.table(silver_results_table)
    .withColumn("session_type", F.lit("RACE"))
    .drop("ingestion_timestamp", "source_file", "race_name", "race_date")
)

sprints_df = (
    spark.read.table(silver_sprints_table)
    .withColumn("session_type", F.lit("SPRINT"))
    .drop("ingestion_timestamp", "source_file", "race_name", "race_date")
)

In [0]:
dim_df = results_df.unionByName(sprints_df)

In [0]:
fact_session_results_df = dim_df.withColumns(
    {
        "is_win": F.when(F.col("finish_position") == 1, True).otherwise(False),
        "is_podium": F.when(F.col("finish_position").between(1, 3), True).otherwise(
            False
        ),
        "has_ponits": F.when(F.col("points") > 0, True).otherwise(False),
    }
)

In [0]:
fact_session_results_df.write.mode("overwrite").format("delta").saveAsTable(
    target_table
)

In [0]:
%sql
select
  *
from
  formula1.gold.fact_results